# Pipeline Test

Verifies the pipeline DAG builder: node creation, edge wiring, serialization, and viewer integration.

In [ ]:
import json
import pathlib

import megane
from megane.pipeline import (
    AddBonds,
    AddLabels,
    AddPolyhedra,
    Filter,
    LoadStructure,
    LoadTrajectory,
    LoadVector,
    Modify,
    Pipeline,
    VectorOverlay,
)

FIXTURES = (pathlib.Path(".").resolve().parent / "fixtures")
assert FIXTURES.exists(), f"Fixtures not found: {FIXTURES}"
print(f"megane v{megane.__version__}")

## Basic pipeline: LoadStructure → Filter → Modify + AddBonds

In [ ]:
pipe = Pipeline()
s = pipe.add_node(LoadStructure(str(FIXTURES / "1crn.pdb")))
f = pipe.add_node(Filter(query="element == 'C'"))
m = pipe.add_node(Modify(scale=1.3))
bonds = pipe.add_node(AddBonds(source="distance"))

pipe.add_edge(s, f)
pipe.add_edge(f, m)
pipe.add_edge(s, bonds)

d = pipe.to_dict()
assert d["version"] == 3, f"Expected version 3, got {d['version']}"
node_types = [n["type"] for n in d["nodes"]]
assert "load_structure" in node_types
assert "filter" in node_types
assert "modify" in node_types
assert "add_bond" in node_types
assert "viewport" in node_types
print(f"Pipeline nodes: {node_types}")
print(f"Edges: {len(d['edges'])}")

## Apply pipeline to viewer

In [ ]:
viewer = megane.MolecularViewer()
viewer.set_pipeline(pipe)

assert viewer._pipeline_enabled is True, "Pipeline not enabled"
pipeline_config = json.loads(viewer._pipeline_json)
assert pipeline_config["version"] == 3
assert len(viewer._node_snapshots_data) > 0, "No node snapshot data"
print(f"Pipeline enabled, {len(viewer._node_snapshots_data)} node snapshots")
viewer

## Pipeline with AddLabels and AddPolyhedra

In [ ]:
pipe2 = Pipeline()
s2 = pipe2.add_node(LoadStructure(str(FIXTURES / "1crn.pdb")))
labels = pipe2.add_node(AddLabels(source="element"))
poly = pipe2.add_node(AddPolyhedra(center_elements=[8]))

pipe2.add_edge(s2, labels)
pipe2.add_edge(s2, poly)

d2 = pipe2.to_dict()
node_types2 = [n["type"] for n in d2["nodes"]]
assert "label_generator" in node_types2
assert "polyhedron_generator" in node_types2
print(f"Labels + Polyhedra pipeline: {node_types2}")

v2 = megane.MolecularViewer()
v2.set_pipeline(pipe2)
v2

## Pipeline with trajectory

In [ ]:
pipe3 = Pipeline()
s3 = pipe3.add_node(LoadStructure(str(FIXTURES / "caffeine_water.pdb")))
t3 = pipe3.add_node(LoadTrajectory(xtc=str(FIXTURES / "caffeine_water_vibration.xtc")))
pipe3.add_edge(s3, t3)

assert len(pipe3._trajectories) > 0, "No trajectories loaded"
print(f"Trajectories: {len(pipe3._trajectories)}")

v3 = megane.MolecularViewer()
v3.set_pipeline(pipe3)
assert v3.total_frames > 0, f"Expected frames > 0, got {v3.total_frames}"
print(f"Total frames: {v3.total_frames}")
v3

## Pipeline with LoadVector and VectorOverlay

In [ ]:
pipe4 = Pipeline()
s4 = pipe4.add_node(LoadStructure(str(FIXTURES / "1crn.pdb")))
vec = pipe4.add_node(LoadVector(str(FIXTURES / "demo_vectors.vec")))
overlay = pipe4.add_node(VectorOverlay(scale=2.0))

pipe4.add_edge(vec, overlay)

d4 = pipe4.to_dict()
node_types4 = [n["type"] for n in d4["nodes"]]
assert "load_vector" in node_types4
assert "vector_overlay" in node_types4
print(f"Vector pipeline: {node_types4}")